# Pilot Assignment with AMPL in Python
## AMPL Modeling for Book Problem 2.6

This notebook solves the same pilot assignment problem with AMPL from Python using `amplpy`.

We solve the base model from book section `2.6` and verify the textbook assignment.


## Requirements

This notebook uses `amplpy` together with the HiGHS solver module.

If your environment does not have `amplpy`, install it first:

```python
%pip install amplpy
```

The first call to `ampl_notebook(modules=["highs"], license_uuid="default")` may download the solver module if it is not already available.


In [1]:
from __future__ import annotations

from functools import wraps
from time import perf_counter


In [2]:
%pip install amplpy
try:
    from amplpy import ampl_notebook
except ImportError as exc:
    raise ImportError(
        "This notebook requires amplpy. Install it with `%pip install amplpy` before running."
    ) from exc


def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = func(*args, **kwargs)
        elapsed = perf_counter() - start
        return result, elapsed
    return wrapper


def create_ampl_instance(solver: str = "highs"):
    return ampl_notebook(modules=[solver], license_uuid="default")


def extract_solution(variable, pilots, crafts):
    values = variable.get_values().to_dict()
    result = {}
    for pilot in pilots:
        chosen_craft = None
        for craft in crafts:
            value = values.get((pilot, craft), values.get((pilot, craft,)))
            if int(round(value)) == 1:
                chosen_craft = craft
                break
        result[pilot] = chosen_craft
    return result


Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: /Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Problem: Pilot Assignment

Each pilot must be assigned to exactly one aircraft, and each aircraft must be used exactly once.
The objective is to maximize the total skill score from Table `2.3`.


In [3]:
PILOT_DATA = ["pilot_1", "pilot_2", "pilot_3", "pilot_4"]
CRAFT_DATA = ["dirigible", "airplane", "jet", "helicopter"]
SCORE_DATA = {
    ("pilot_1", "dirigible"): 6,
    ("pilot_1", "airplane"): 2,
    ("pilot_1", "jet"): 8,
    ("pilot_1", "helicopter"): 5,
    ("pilot_2", "dirigible"): 9,
    ("pilot_2", "airplane"): 3,
    ("pilot_2", "jet"): 5,
    ("pilot_2", "helicopter"): 8,
    ("pilot_3", "dirigible"): 4,
    ("pilot_3", "airplane"): 8,
    ("pilot_3", "jet"): 3,
    ("pilot_3", "helicopter"): 4,
    ("pilot_4", "dirigible"): 6,
    ("pilot_4", "airplane"): 7,
    ("pilot_4", "jet"): 6,
    ("pilot_4", "helicopter"): 4,
}

EXPECTED = {
    "pilot_1": "jet",
    "pilot_2": "helicopter",
    "pilot_3": "airplane",
    "pilot_4": "dirigible",
    "skill_score": 30,
}


In [4]:
@timed
def solve_pilot_assignment_with_ampl(pilots, crafts, scores, solver: str = "highs"):
    ampl = create_ampl_instance(solver)
    ampl.eval(
        r'''
        set PILOTS ordered;
        set CRAFTS ordered;
        param score {PILOTS, CRAFTS};

        var Assign {p in PILOTS, c in CRAFTS} binary;

        maximize TotalScore:
            sum {p in PILOTS, c in CRAFTS} score[p, c] * Assign[p, c];

        subject to OneCraftPerPilot {p in PILOTS}:
            sum {c in CRAFTS} Assign[p, c] = 1;

        subject to OnePilotPerCraft {c in CRAFTS}:
            sum {p in PILOTS} Assign[p, c] = 1;
        '''
    )
    ampl.set["PILOTS"] = pilots
    ampl.set["CRAFTS"] = crafts
    ampl.param["score"] = scores
    ampl.solve(solver=solver)

    solution = extract_solution(ampl.get_variable("Assign"), pilots, crafts)
    solution["skill_score"] = int(round(ampl.get_objective("TotalScore").value()))
    return solution


In [5]:
ampl_result, ampl_time = solve_pilot_assignment_with_ampl(
    pilots=PILOT_DATA,
    crafts=CRAFT_DATA,
    scores=SCORE_DATA,
)

print("=== PILOT ASSIGNMENT RESULTS WITH AMPL ===")
print(f"Solution -> {ampl_result}")
print(f"Time     -> {ampl_time:.8f} seconds")

assert ampl_result == EXPECTED
print("AMPL matches the textbook optimum.")


/Users/lfuentesl/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


AMPL Version 20250901 (Darwin-22.6.0, 64-bit)
Demo license with maintenance expiring 20270131.
Using license file "/Users/lfuentesl/Library/Python/3.9/lib/python/site-packages/ampl_module_base/bin/ampl.lic".

HiGHS 1.11.0: HiGHS 1.11.0: optimal solution; objective 30
7 simplex iterations
1 branching nodes
=== PILOT ASSIGNMENT RESULTS WITH AMPL ===
Solution -> {'pilot_1': 'jet', 'pilot_2': 'helicopter', 'pilot_3': 'airplane', 'pilot_4': 'dirigible', 'skill_score': 30}
Time     -> 1.72933592 seconds
AMPL matches the textbook optimum.


## Variants in the Book

Section `2.6` does not propose an additional student variation after the base model, so this AMPL notebook closes with the reference assignment only.
